# Taller: Ingeniería de prompts con IA

> **Clase 2 · El arte del prompt**

En este taller vas a probar **en vivo** cómo cambia la respuesta de una IA según cómo escribas el pedido. Usamos **OpenRouter**: una sola API que te da acceso a muchos modelos, sin instalar nada local.

## Qué vas a experimentar

1. **Zero-shot**: pedir una tarea sin dar ejemplos.
2. **Marcos de prompting**: dar rol, contexto y formato (anatomía del prompt).
3. **Chain of Thought (CoT)**: pedirle que piense paso a paso.
4. **System prompt vs. prompt de usuario**: la diferencia entre *cómo se comporta* y *qué se le pide*.

> 💡 **Regla del taller:** cambiá **una sola cosa por vez** y observá qué pasó. Así sabés qué generó cada diferencia.

La idea sigue la del módulo de Prompt Engineering: el wrapper `llamar_llm()` separa el `system_prompt` (comportamiento) del `prompt` (pedido del usuario).

---
## 1. Obtené tu API key de OpenRouter

1. Andá a [openrouter.ai](https://openrouter.ai) y creá una cuenta (gratis).
2. En **Keys** → **Create key**, generá una clave.
3. Varias herramientas ofrecen crédito inicial gratis para probar.

> ⚠️ **Cuidá tu clave.** Usá modelos de bajo costo y textos ficticios. No pongas datos personales ni confidenciales.

In [ ]:
import getpass

# Te la pide una sola vez por sesión. No queda guardada en el notebook.
OPENROUTER_API_KEY = getpass.getpass("Pegá tu API key de OpenRouter: ")
print("Clave cargada en esta sesión. ✓")

Clave cargada en esta sesión. ✓


---
## 2. Conectarse y armar el wrapper

Este bloque usa el mismo patrón del curso: una función única `llamar_llm()`. Recibe:

- `prompt`: el pedido del usuario (lo que vas a cambiar en cada experimento).
- `system_prompt`: cómo debe comportarse el asistente.
- `temperature`: qué tan creativo o conservador es (0 = casi fijo, 1 = más suelto).

El modelo elegido corre en OpenRouter y no necesita instalación. Necesitás internet.

In [2]:
import ssl
import json
from urllib.request import Request, urlopen
import certifi

OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"
# Modelo barato y bueno para probar prompts. Lo podés cambiar.
MODELO = "openai/gpt-4o-mini"
CONTEXTO_SSL = ssl.create_default_context(cafile=certifi.where())

def llamar_llm(prompt, system_prompt="Responde en español, simple y breve.", temperature=0.3, max_tokens=200):
    payload = {
        "model": MODELO,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": prompt},
        ],
        "temperature": temperature,
        "max_tokens": max_tokens,
    }
    req = Request(
        OPENROUTER_URL,
        data=json.dumps(payload).encode("utf-8"),
        headers={
            "Authorization": f"Bearer {OPENROUTER_API_KEY}",
            "Content-Type": "application/json",
        },
    )
    with urlopen(req, context=CONTEXTO_SSL) as resp:
        datos = json.loads(resp.read().decode("utf-8"))
    return datos["choices"][0]["message"]["content"].strip()

print("Wrapper listo. Probemos la conexión:")
print(llamar_llm("Decí solo: 'Todo listo'", max_tokens=15))

ModuleNotFoundError: No module named 'certifi'

---
## 3. Zero-shot: la base

**Zero-shot** es darle la tarea **sin ningún ejemplo**. Es el punto de partida de casi todo prompt. Probá la misma tarea con prompts cada vez más específicos y mirá cómo cambia la respuesta.

In [ ]:
# Versión 1: pedido mínimo
pedido_minimo = "Explicá qué es un prompt."
print("PEDIDO MÍNIMO:")
print("-" * 40)
print(llamar_llm(pedido_minimo))
print()

# Versión 2: mismo tema, con formato pedido
pedido_formato = "Explicá qué es un prompt en exactamente 3 puntos breves."
print("CON FORMATO:")
print("-" * 40)
print(llamar_llm(pedido_formato))

> 💡 **Observá:** mismo tema, distinta respuesta. El pedido mínimo puede irse por las ramas; pedir un formato lo enfoca.

**Probá vos:** cambiá el tema por algo de tu interés y volvé a correr la celda.

---
## 4. Marcos de prompting: rol + contexto + formato

Un **marco** (framework) es una plantilla para estructurar el pedido. El más usado combina **rol**, **contexto**, **tarea** y **formato**. Comparalo contra el zero-shot de arriba.

In [ ]:
prompt_enmarcado = """Sos un docente de informática para estudiantes que recién empiezan.
Contexto: estás preparando una guía breve para una clase.
Tarea: explicá qué es un prompt.
Formato: respondé en 3 viñetas cortas, con un ejemplo al final.
"""
print("PROMPT ENMARCADO (rol + contexto + tarea + formato):")
print("-" * 40)
print(llamar_llm(prompt_enmarcado))

> 💡 El modelo no sabe *quién* sos ni *qué necesitás*. Darle rol y contexto le da el marco para responder mejor.

**Probá vos:** cambiá el rol (por ejemplo, "vos sos un amigo que explica simple") y fijate cómo cambia el tono.

---
## 5. Few-shot: enseñar con ejemplos

**Few-shot** es mostrarle **ejemplos** del resultado que esperás. No le explicamos *cómo* hacerlo: se lo mostramos. Compará la consistencia con el zero-shot.

In [ ]:
mis_ejemplos = """Clasificá el tono de cada comentario como Positivo, Negativo o Neutro.

Comentario: "El producto llegó antes de lo esperado y funciona perfecto."
Tono: Positivo

Comentario: "Tardó tres semanas y llegó roto."
Tono: Negativo

Comentario: "[:]"
"""

# El alumno reemplaza el "[:]" por un tercer comentario y prueba.
comentario_nuevo = "Sirve, cumple lo que dice."   # cambiá este comentario
prompt_few_shot = mis_ejemplos.replace("[:]", comentario_nuevo)

print("FEW-SHOT: mismo patrón, entrada nueva")
print("-" * 40)
print(llamar_llm(prompt_few_shot))

> 💡 Los ejemplos le enseñan también el **formato** de salida, no solo el concepto. Con pocos ejemplos bien elegidos alcanza.

**Probá vos:** cambiá el `comentario_nuevo` por otro y observá cómo repite el patrón de respuesta.

---
## 6. Chain of Thought: pensá paso a paso

Para tareas de razonamiento, pedirle que **piense paso a paso antes de responder** suele mejorar el resultado y te deja ver **dónde** se equivoca.

In [ ]:
problema = "Una campera cuesta $6000 y tiene 25% de descuento. ¿Cuánto pagás?"

print("RESPUESTA DIRECTA (sin CoT):")
print("-" * 40)
print(llamar_llm(problema, max_tokens=120))
print()

pedido_con_cot = problema + "\n\nPensá paso a paso antes de dar la respuesta final."
print("CON CHAIN OF THOUGHT:")
print("-" * 40)
print(llamar_llm(pedido_con_cot, max_tokens=220))

> 💡 **CoT** no sirve para todo: en tareas simples suma ruido. Se nota la diferencia cuando hay **varios pasos o criterios**.

**Probá vos:** cambiá el problema por uno con dos o tres pasos y compará ambas versiones.

---
## 7. System prompt vs. prompt de usuario

Son dos mensajes distintos:

- **System prompt**: define *cómo se comporta el asistente* (rol, tono, reglas). Es un mensaje de sistema.
- **Prompt de usuario**: es *lo que pedís* en cada consulta.

Probemos el mismo pedido con dos system prompts distintos: el modelo cambia de personalidad sin tocar la pregunta.

In [ ]:
la_pregunta = "¿Qué recomendarías comer hoy?"

system_estricto = "Respondé en una sola oración corta. Sos formal y práctico."
system_creativo = "Respondé con entusiasmo y agregá un toque de humor. Sos relajado y cercano."

print("SYSTEM PROMPT FORMAL:")
print("-" * 40)
print(llamar_llm(la_pregunta, system_prompt=system_estricto, temperature=0.8))
print()
print("SYSTEM PROMPT CREATIVO:")
print("-" * 40)
print(llamar_llm(la_pregunta, system_prompt=system_creativo, temperature=0.8))

> 💡 **La misma pregunta, distinta personalidad.** Ese es el poder de separar el system prompt del pedido: definís el comportamiento de una vez y la pregunta cambia sin reescribir todo.

**Probá vos:** escribí tu propio system prompt (por ejemplo, "sos un tutor paciente que explica simple") y fijate cómo responde tu pregunta.

---
## 8. Cierre y reflexión

Con lo que probaste, respondé en vos mismo o en el grupo:

1. ¿Qué cambió más: el **formato** pedido o el **rol**?
2. ¿En qué casos te sirvió el **few-shot** por sobre el zero-shot?
3. ¿Cuándo conviene el **chain of thought**?
4. ¿Por qué conviene separar el **system prompt** del pedido?

> **Recordá:** la IA es una herramienta. La respuesta puede sonar convincente y aun así ser imprecisa: revisá, probá y decidí con criterio.